In [1]:
!pip install optuna

  Using cached optuna-4.8.0-py3-none-any.whl.metadata (17 kB)
  Using cached colorlog-6.10.1-py3-none-any.whl.metadata (11 kB)
Using cached optuna-4.8.0-py3-none-any.whl (419 kB)
Using cached colorlog-6.10.1-py3-none-any.whl (11 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [optuna]2m1/2 [optuna]


In [1]:
import pandas as pd
import pickle
import numpy as np 

import optuna 
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score

from sklearn.metrics import accuracy_score, f1_score, classification_report

/opt/anaconda3/envs/mlflow3/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [11]:
link = '/Users/maxkucher/bcs_ai/intent_model/intent_data.csv'
data = pd.read_csv(link)
data

,sequence_id,t,X,Y,vx,vy,speed,motion_class,distance,threat,intent
0,0,0,-60.946138,196.966037,-4.469285,-0.450112,4.491894,-1,206.179658,0.127225,reposition
1,0,1,-55.693151,197.403167,5.252986,0.437131,5.271143,1,205.109087,0.128595,reposition
2,0,2,-60.620444,196.570349,-4.927292,-0.832818,4.997179,-1,205.705470,0.127830,reposition
3,0,3,-65.290447,196.784927,-4.670004,0.214578,4.674931,-1,207.333428,0.125766,reposition
4,0,4,-70.507729,197.183724,-5.217282,0.398797,5.232501,-1,209.410509,0.123180,reposition
...,...,...,...,...,...,...,...,...,...,...,...
49995,4999,5,35.331296,115.564848,0.611553,0.556272,0.826702,1,120.845085,0.298660,idle
49996,4999,6,35.642107,115.796848,0.310811,0.232000,0.387849,1,121.158037,0.297726,idle
49997,4999,7,36.173334,115.432187,0.531227,-0.364662,0.644344,1,120.967350,0.298295,idle
49998,4999,8,36.379956,115.689068,0.206622,0.256881,0.329667,2,121.274324,0.297380,idle


In [9]:
# columns_to_drop = ['sequence_id']
# data = data.drop(columns_to_drop, axis='columns')
# data

In [12]:
data['intent'].value_counts()

intent
idle          12990
retreat       12650
attack        12390
reposition    11970
Name: count, dtype: int64

In [13]:
INTENT_MAP = {
    "attack": 0,
    "retreat": 1,
    "reposition": 2,
    "idle": 3
}

data["intent_encoded"] = data["intent"].map(INTENT_MAP)

In [14]:
data = data.drop('intent', axis='columns')
data

,sequence_id,t,X,Y,vx,vy,speed,motion_class,distance,threat,intent_encoded
0,0,0,-60.946138,196.966037,-4.469285,-0.450112,4.491894,-1,206.179658,0.127225,2
1,0,1,-55.693151,197.403167,5.252986,0.437131,5.271143,1,205.109087,0.128595,2
2,0,2,-60.620444,196.570349,-4.927292,-0.832818,4.997179,-1,205.705470,0.127830,2
3,0,3,-65.290447,196.784927,-4.670004,0.214578,4.674931,-1,207.333428,0.125766,2
4,0,4,-70.507729,197.183724,-5.217282,0.398797,5.232501,-1,209.410509,0.123180,2
...,...,...,...,...,...,...,...,...,...,...,...
49995,4999,5,35.331296,115.564848,0.611553,0.556272,0.826702,1,120.845085,0.298660,3
49996,4999,6,35.642107,115.796848,0.310811,0.232000,0.387849,1,121.158037,0.297726,3
49997,4999,7,36.173334,115.432187,0.531227,-0.364662,0.644344,1,120.967350,0.298295,3
49998,4999,8,36.379956,115.689068,0.206622,0.256881,0.329667,2,121.274324,0.297380,3


In [15]:
def prepare_sequences(df, seq_len=10):
    X = []
    y = []

    for _, group in df.groupby("sequence_id"):
        group = group.sort_values("t")

        features = group[[
            "X", "Y", "vx", "vy",
            "speed", "motion_class",
            "distance", "threat"
        ]].values

        if len(features) == seq_len:
            X.append(features.flatten())  
            y.append(group["intent_encoded"].iloc[0])

    return np.array(X), np.array(y)

In [16]:
x, y = prepare_sequences(data)

In [21]:
x.shape

(5000, 80)

In [ ]:
# x = data.drop('intent_encoded', axis='columns')
# y = data['intent_encoded']

In [22]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2)

In [23]:
def objective(trial):
    model = RandomForestClassifier(
        n_estimators=trial.suggest_int("n_estimators", 50, 300),
        max_depth=trial.suggest_int("max_depth", 2, 32),
        min_samples_split=trial.suggest_int("min_samples_split", 2, 20),
        min_samples_leaf=trial.suggest_int("min_samples_leaf", 1, 20),
        max_features=trial.suggest_categorical("max_features", ["sqrt", "log2", None]),
        n_jobs=-1,
        random_state=42
    )

    score = cross_val_score(model, x_train, y_train)

    return score.mean()

In [24]:
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50)

[I 2026-04-05 13:37:55,928] A new study created in memory with name: no-name-4a8c2741-670f-4862-a9c1-b33df251534e
[I 2026-04-05 13:37:57,715] Trial 0 finished with value: 1.0 and parameters: {'n_estimators': 81, 'max_depth': 2, 'min_samples_split': 16, 'min_samples_leaf': 5, 'max_features': None}. Best is trial 0 with value: 1.0.
[I 2026-04-05 13:37:59,357] Trial 1 finished with value: 1.0 and parameters: {'n_estimators': 258, 'max_depth': 23, 'min_samples_split': 7, 'min_samples_leaf': 9, 'max_features': 'log2'}. Best is trial 0 with value: 1.0.
[I 2026-04-05 13:38:00,400] Trial 2 finished with value: 1.0 and parameters: {'n_estimators': 146, 'max_depth': 19, 'min_samples_split': 13, 'min_samples_leaf': 15, 'max_features': 'log2'}. Best is trial 0 with value: 1.0.
[I 2026-04-05 13:38:02,231] Trial 3 finished with value: 1.0 and parameters: {'n_estimators': 269, 'max_depth': 27, 'min_samples_split': 10, 'min_samples_leaf': 16, 'max_features': 'sqrt'}. Best is trial 0 with value: 1.0.
[

In [25]:
best_model = RandomForestClassifier(**study.best_params)
best_model.fit(x_train, y_train)

,n_estimators,81
,criterion,'gini'
,max_depth,2
,min_samples_split,16
,min_samples_leaf,5
,min_weight_fraction_leaf,0.0
,max_features,None
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [26]:
preds = best_model.predict(x_test)
accuracy = accuracy_score(y_test, preds)
f1 = f1_score(y_test, preds, average="weighted")

print(f"Accuracy: {accuracy} | F1-score: {f1}")

Accuracy: 1.0 | F1-score: 1.0


In [27]:
with open("intent_model.pkl", "wb") as f:
    pickle.dump(best_model, f)